# FarmAdvocate: Colab runner

Runs the GPU checks and the full evaluation, then uploads the results to a private dataset in your Hugging Face account.

1. **Runtime > Change runtime type > T4 GPU**
2. Click the key icon on the left (Secrets), add `HF_TOKEN` with your Hugging Face **write** token, and switch on notebook access. The token stays in Colab Secrets and is never written into this notebook.
3. **Runtime > Run all.** Allow access to the secret when asked. The AgriParam download is 11.4 GB, so the first run takes a while.

The last cell prints `UPLOADED` when everything is saved.

In [ ]:
import os, sys
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Python', sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!rm -rf FarmAdvocate && git clone -q https://github.com/mohanchandrass/FarmAdvocate.git
!pip install -q uv
!uv venv -q --python 3.12 --allow-existing /content/venv
!VIRTUAL_ENV=/content/venv uv pip install -q -r FarmAdvocate/requirements.txt
!/content/venv/bin/python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

In [ ]:
# Gate 1: translation on GPU, then AgriParam with real weights (quality, speed, peak memory).
!mkdir -p /content/FarmAdvocate/eval/results
!cd /content/FarmAdvocate && /content/venv/bin/python scripts/gate1_translation.py eval/results/gate1a_gpu.json pr 2>&1 | tail -3
!cd /content/FarmAdvocate && /content/venv/bin/python scripts/gate1_agriparam.py eval/results/gate1b.json > eval/results/gate1b.log 2>&1; tail -4 eval/results/gate1b.log

In [ ]:
# Full evaluation: 50 cases through the pipeline plus the closed-book baseline (about 30-60 minutes).
!cd /content/FarmAdvocate && /content/venv/bin/python eval/run_eval.py full 2>&1 | grep -v -iE 'warn|^$' | tail -15
!cd /content/FarmAdvocate && /content/venv/bin/python eval/run_eval.py retrieval > eval/results/retrieval.log 2>&1
!cd /content/FarmAdvocate && /content/venv/bin/python eval/run_eval.py metrics

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
repo = 'DaMCSS/farmadvocate-results'
api.create_repo(repo, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(folder_path='/content/FarmAdvocate/eval/results', repo_id=repo, repo_type='dataset')
print('UPLOADED to', repo)